> 在线讲义：[第 6 章 · A/B 判定规则与多重比较：先定规则再测，别偷看，比多了要校正](https://wikieden.github.io/math-and-eval-stats/lessons/06-ab-rules-multiple-comparisons.html) · 本书《数学与实验统计》

# 第 6 章 · A/B 判定规则与多重比较：先定规则再测，别偷看，比多了要校正

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/06-ab-rules-multiple-comparisons.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 偷看 | 模拟 | `peeking_false_positive()` | 0.188 / 0.049 |
| 第二步 FWER | 1 − 0.95^k | `family_error(20)` | 0.642 |
| 第二步 模拟 | 10 个任务 | `null_tasks_significant()` | 平均 0.52 个 |
| 第三步 校正 | Bonferroni / Holm | `bonferroni(P_VALUES)`、`holm(P_VALUES)` | 检出 1 / 3 个 |
| 第四步 IQM | 中间 50% | `iqm(TASK_SCORES)` | 0.487 |
| 第五步 聚类 | 式 (4) | `clustered_se(clusters)` | 0.1095 → 0.1817 |

例子：偷看的模拟（两个策略其实一样，都是 0.7）；编的 6 个 p 值做 Holm；编的任务分数算 IQM；Miller 式 (4) 的聚类 SE。
里程碑 ME-M5 的"20 个任务"模拟留给自己做，这里用 10 个。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
from scipy.stats import norm
P_VALUES = [0.004, 0.009, 0.012, 0.040, 0.130, 0.470]  # 编的 6 个比较
TASK_SCORES = np.array([0.02, 0.10, 0.35, 0.41, 0.48, 0.52, 0.55, 0.61, 0.70, 0.98])  # 编的 10 个任务分数

**函数 `family_error`**：k 个独立检验、每个都在 α 水平上做，至少误报一个的概率 1 − (1 − α)^k。

In [ ]:
def family_error(k: int, alpha: float = 0.05) -> float:
    """k 个独立检验、每个都在 α 水平上做，至少误报一个的概率 1 − (1 − α)^k。"""
    return 1 - (1 - alpha) ** k

**函数 `bonferroni`**：Bonferroni：每个检验用 α / m。

In [ ]:
def bonferroni(pvals, alpha: float = 0.05) -> list[bool]:
    """Bonferroni：每个检验用 α / m。"""
    m = len(pvals)
    return [p < alpha / m for p in pvals]

**函数 `holm`**：Holm：p 值从小到大，第 i 个（从 0 数）和 α / (m − i) 比，一旦不过就停。

In [ ]:
def holm(pvals, alpha: float = 0.05) -> list[bool]:
    """Holm：p 值从小到大，第 i 个（从 0 数）和 α / (m − i) 比，一旦不过就停。"""
    m = len(pvals)
    order = np.argsort(pvals)
    reject = [False] * m
    for i, j in enumerate(order):
        if pvals[j] < alpha / (m - i):
            reject[j] = True
        else:
            break
    return reject

**函数 `peeking_false_positive`**：两个策略其实一样（都是 p）。每跑 every 次就做一次 z 检验，一旦 p < 0.05 就宣布"B 更好"。返回 (偷看时的误报率, 只在最后看一次的误报率)。

In [ ]:
def peeking_false_positive(p: float = 0.7, n_max: int = 100, every: int = 10, reps: int = 20000, seed: int = 0) -> tuple[float, float]:
    """两个策略其实一样（都是 p）。每跑 every 次就做一次 z 检验，一旦 p < 0.05 就宣布"B 更好"。返回 (偷看时的误报率, 只在最后看一次的误报率)。"""
    rng = np.random.default_rng(seed)
    a = rng.random((reps, n_max)) < p
    b = rng.random((reps, n_max)) < p
    hit_any = np.zeros(reps, bool)
    hit_last = np.zeros(reps, bool)
    for n in range(every, n_max + 1, every):
        ka, kb = a[:, :n].sum(1), b[:, :n].sum(1)
        pbar = (ka + kb) / (2 * n)
        se = np.sqrt(pbar * (1 - pbar) * 2 / n)
        z = np.divide(kb - ka, n * se, out=np.zeros(reps), where=se > 0)
        sig = np.abs(z) > 1.959964
        hit_any |= sig
        if n == n_max:
            hit_last = sig
    return float(hit_any.mean()), float(hit_last.mean())

**函数 `null_tasks_significant`**：k 个任务、两个模型其实一样：平均每轮有几个任务"显著"（不校正）。

In [ ]:
def null_tasks_significant(k: int = 10, n: int = 50, p: float = 0.6, reps: int = 5000, seed: int = 1) -> float:
    """k 个任务、两个模型其实一样：平均每轮有几个任务"显著"（不校正）。"""
    rng = np.random.default_rng(seed)
    ka, kb = rng.binomial(n, p, (reps, k)), rng.binomial(n, p, (reps, k))
    pbar = (ka + kb) / (2 * n)
    se = np.sqrt(pbar * (1 - pbar) * 2 / n)
    z = np.divide(kb - ka, n * se, out=np.zeros((reps, k)), where=se > 0)
    return float((np.abs(z) > 1.959964).sum(1).mean())

**函数 `iqm`**：四分位间均值：排序后去掉最低 25% 和最高 25%，剩下的取平均。

In [ ]:
def iqm(x: np.ndarray) -> float:
    """四分位间均值：排序后去掉最低 25% 和最高 25%，剩下的取平均。"""
    x = np.sort(np.asarray(x))
    n = len(x)
    lo, hi = int(math.floor(n * 0.25)), int(math.ceil(n * 0.75))
    return float(x[lo:hi].mean())

**函数 `clustered_se`**：Miller 2024 式 (4)：SE_clustered = √(SE_CLT² + (1/n²) Σ_c Σ_i Σ_{j≠i} (s_i − s̄)(s_j − s̄))。返回 (普通 SE, 聚类 SE)。

In [ ]:
def clustered_se(scores_by_cluster: list[list[float]]) -> tuple[float, float]:
    """Miller 2024 式 (4)：SE_clustered = √(SE_CLT² + (1/n²) Σ_c Σ_i Σ_{j≠i} (s_i − s̄)(s_j − s̄))。返回 (普通 SE, 聚类 SE)。"""
    allv = np.concatenate([np.asarray(c, float) for c in scores_by_cluster])
    n, sbar = len(allv), allv.mean()
    se_clt = math.sqrt(allv.var(ddof=0) / n)
    cross = 0.0
    for c in scores_by_cluster:
        d = np.asarray(c, float) - sbar
        cross += d.sum() ** 2 - (d**2).sum()
    return se_clt, math.sqrt(se_clt**2 + cross / n**2)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 偷看：两个策略一样（都是 0.7），每 10 次看一眼，最多 100 次")
peek, last = peeking_false_positive()
print(f"  每 10 次看一次、显著就停：误报率 {peek:.3f}；只在第 100 次看一次：{last:.3f}")

In [ ]:
print("\n第二步 同时比很多个：至少误报一个的概率")
for k in (1, 5, 10, 20):
    print(f"  {k:2d} 个检验：{family_error(k):.3f}")
print(f"  模拟：10 个任务、两模型一样、每任务 50 次，平均 {null_tasks_significant():.2f} 个任务\"显著\"")

In [ ]:
print("\n第三步 Bonferroni 和 Holm（6 个 p 值）")
print(f"  p 值：{P_VALUES}")
print(f"  不校正：{[p < 0.05 for p in P_VALUES]}")
print(f"  Bonferroni（α/6 = {0.05 / 6:.4f}）：{bonferroni(P_VALUES)}")
print(f"  Holm：{holm(P_VALUES)}")

In [ ]:
print("\n第四步 多任务汇总：均值、中位数、IQM")
print(f"  10 个任务：均值 {TASK_SCORES.mean():.3f}，中位数 {np.median(TASK_SCORES):.3f}，IQM {iqm(TASK_SCORES):.3f}")

In [ ]:
print("\n第五步 题目成组出现：聚类标准误（Miller 式 4）")
clusters = [[1, 1, 1, 1], [0, 0, 0, 0], [1, 1, 1, 0], [0, 0, 1, 0], [1, 1, 1, 1]]
a, b = clustered_se(clusters)
print(f"  5 篇文章各 4 道题：普通 SE {a:.4f}，聚类 SE {b:.4f}（{b / a:.2f} 倍）")

## 3. 代码里才看得到的

1. **交叉项的小技巧**：$\sum_{i\ne j}d_id_j = (\sum d)^2 - \sum d^2$，不用写双重循环。
2. **IQM 在样本数不能被 4 整除时的取法**：10 个数去掉"最低 25%"是 2.5 个，代码取 `floor` 和 `ceil`，实际去掉两头各 2 个、用中间 6 个；rliable 用的是按比例截尾平均，结果略有差别。
3. **偷看模拟是向量化的**：一次生成 2 万轮 × 100 次的结果矩阵，每 10 列做一次检验，`hit_any |= sig` 记录"有没有哪次显著过"。

## 4. 自己改一改

1. 把 `peeking_false_positive` 的 `every` 从 10 改成 1（每测一次就看），误报率涨到多少？
2. 把 `clusters` 里每篇文章的 4 道题都改成一半对一半错，聚类 SE 和普通 SE 还差多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先写规则再开测，偷看一次多一分误报；
> 比得越多越容易撞上，一减零九五的 k 次方；
> Bonferroni 除以 m，Holm 从小到大逐个放宽；
> 多任务报 IQM 加区间，题目成组用聚类标准误。